In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Inversion fallback: minimal paper-experiment preparation

This notebook is an executable USER-RUN experiment, not a result. No real
experiment has been run during implementation. It is initial-noise watermarking,
not a trajectory watermark. Select GPU and Run all only when ready to execute.

Fixed new calibration: two OFF sources (green tractor/wheat field seed20261101,
black rowing boat/canal seed20261102). Evaluation: brown horse/fence seed20261201
and orange balloon/hills seed20261202, each OFF + STATE_A/B + STATIC_A/B.
Total12 sources,36 views. STATIC repeats +/-initial direction with identical
pad/channel/support/magnitude, but code distances differ: temporal-pattern
ablation, not isolated dynamic-observer evidence.

Views: actual full181 MP4 and lossless129 RGB-MP4 crops16/17. Full view is
inverted as46 latent slices, then fixed first33 are scored with unchanged31-slice
raw evidence; this is not whole-video evidence aggregation. Crops invert33.
STATE/STATIC decoders each use14shift and shift0 scores. Observer is auxiliary.

Thresholds use ONLY the two new calibration OFF sources: maximum score over3
views per source, then maximum over2 sources, independently for all4channels.
Strict score>threshold. Freeze threshold.json and its hash BEFORE any eval
launch. Missing calibration leaves UNCALIBRATED, while eval rankings still run.
No tuning, positive-data thresholds, previously viewed holdout, replacements or
per-video parameter selection. OFF false accepts and marked all-view recovery
are clustered by source; retain missing/partial and all view/candidate records.
Two calibration and two test negative sources cannot support low-FPR claims.

Budget:1200 forward Transformer calls/600steps,3600 inverse Transformer calls/
1800updates,12VAEdecode/36encode,12MP4save+24lossless crop saves. Model lifetimes
are separate. Full/crop quality uses same-seed OFF; PSNR is not perceptual PASS.
Inspect content, artifacts and motion manually. Costs/resources/failures persist.

Outputs:MyDrive/Video-WM/InversionPaperPreparation/inversion_paper_<UTC> with
source.zip, launcher/case logs, calibration/evaluation, threshold and provenance.
CPU/synthetic preparation tests only; no actual new data have been scored.
Source SHA:7fa72efea8a6ef04c2e62748e9fcccaeb5549d78.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = '7fa72efea8a6ef04c2e62748e9fcccaeb5549d78'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'inversion_paper_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/InversionPaperPreparation') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.inversion_paper_run', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({'status':result['status'], 'sources':result['source_denominator'], 'views':result['view_denominator'], 'thresholds':result.get('thresholds'), 'summary':result.get('summary'), 'fixed_calls':result['fixed_calls'], 'actual_calls':result.get('actual_calls_observed')}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
